# Bias Phase 2 — Data Analysis & Inter-Annotator Agreement

In [ ]:
import numpy as np
import pandas as pd
import krippendorff
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from scipy.stats import spearmanr, kendalltau
from itertools import combinations
import matplotlib.pyplot as plt
import seaborn as sns

df_conflicts = pd.read_csv('data/annotated_conflicts.csv')
df_merged = pd.read_csv('data/annotated_merged.csv')

# Ordinal scale: Far Left < Left < Center < Right < Far Right
label_order = ['Far Left', 'Left', 'Center', 'Right', 'Far Right']
label_to_int = {l: i for i, l in enumerate(label_order)}

annotators = ['bias_label_praveen', 'bias_label_ashini', 'bias_label_dinithi']

print(f"Conflicts: {len(df_conflicts)} rows")
print(f"Merged (single-annotator): {len(df_merged)} rows")

## 1. Inter-Annotator Agreement (Ordinal)

Since bias labels follow an ordinal scale (Far Left → Left → Center → Right → Far Right), we use metrics that account for the **distance** between labels — a disagreement of Far Left vs Left is less severe than Far Left vs Far Right.

### 1.1 Pairwise Weighted Cohen's Kappa (Quadratic Weights)

In [ ]:
print("=== Pairwise Weighted Cohen's Kappa (quadratic weights) ===\n")
for a1, a2 in combinations(annotators, 2):
    mask = df_conflicts[a1].notna() & df_conflicts[a2].notna()
    subset = df_conflicts[mask]
    name1, name2 = a1.split('_')[-1], a2.split('_')[-1]
    
    # Exact agreement
    exact_agree = (subset[a1] == subset[a2]).mean() * 100
    
    # Weighted kappa (quadratic) — treats ordinal distance
    kappa_w = cohen_kappa_score(
        subset[a1].map(label_to_int), subset[a2].map(label_to_int),
        weights='quadratic', labels=list(range(5))
    )
    
    # Adjacent agreement (exact OR off-by-one)
    diff = (subset[a1].map(label_to_int) - subset[a2].map(label_to_int)).abs()
    adjacent_agree = (diff <= 1).mean() * 100
    
    print(f"{name1} vs {name2} (n={len(subset)}):")
    print(f"  Weighted Kappa (quadratic): {kappa_w:.4f}")
    print(f"  Exact agreement:            {exact_agree:.1f}%")
    print(f"  Adjacent agreement (±1):    {adjacent_agree:.1f}%")
    print()

### 1.2 Krippendorff's Alpha (Ordinal)

Handles missing raters and uses ordinal distance function — the standard multi-rater ordinal IAA metric.

In [ ]:
# Build reliability data matrix for krippendorff (raters × items, NaN for missing)
reliability_data = []
for col in annotators:
    reliability_data.append(
        df_conflicts[col].map(label_to_int).values.astype(float)
    )
reliability_data = np.array(reliability_data)

alpha_ordinal = krippendorff.alpha(reliability_data=reliability_data, level_of_measurement='ordinal')
alpha_nominal = krippendorff.alpha(reliability_data=reliability_data, level_of_measurement='nominal')

print(f"=== Krippendorff's Alpha (all 3 annotators, n={len(df_conflicts)}) ===\n")
print(f"  Ordinal alpha:  {alpha_ordinal:.4f}")
print(f"  Nominal alpha:  {alpha_nominal:.4f}  (for comparison — ignores scale distance)")
print()

# Also compute on the subset where all 3 annotated
mask_all = df_conflicts[annotators].notna().all(axis=1)
rel_all3 = []
for col in annotators:
    rel_all3.append(df_conflicts.loc[mask_all, col].map(label_to_int).values.astype(float))
rel_all3 = np.array(rel_all3)

alpha_all3 = krippendorff.alpha(reliability_data=rel_all3, level_of_measurement='ordinal')
print(f"Ordinal alpha (only articles with all 3 raters, n={mask_all.sum()}): {alpha_all3:.4f}")

### 1.3 Spearman & Kendall Rank Correlations (Pairwise)

In [ ]:
print("=== Pairwise Rank Correlations ===\n")
for a1, a2 in combinations(annotators, 2):
    mask = df_conflicts[a1].notna() & df_conflicts[a2].notna()
    subset = df_conflicts[mask]
    name1, name2 = a1.split('_')[-1], a2.split('_')[-1]
    
    v1 = subset[a1].map(label_to_int)
    v2 = subset[a2].map(label_to_int)
    
    rho, p_rho = spearmanr(v1, v2)
    tau, p_tau = kendalltau(v1, v2)
    
    print(f"{name1} vs {name2} (n={len(subset)}):")
    print(f"  Spearman rho: {rho:.4f}  (p={p_rho:.4e})")
    print(f"  Kendall tau:  {tau:.4f}  (p={p_tau:.4e})")
    print()

### 1.4 Pairwise Confusion Matrices (Ordinal Heatmaps)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (a1, a2) in enumerate(combinations(annotators, 2)):
    mask = df_conflicts[a1].notna() & df_conflicts[a2].notna()
    subset = df_conflicts[mask]
    name1, name2 = a1.split('_')[-1], a2.split('_')[-1]
    
    cm = confusion_matrix(
        subset[a1].map(label_to_int), subset[a2].map(label_to_int),
        labels=list(range(5))
    )
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_order, yticklabels=label_order,
                ax=axes[idx])
    axes[idx].set_title(f'{name1} vs {name2} (n={len(subset)})')
    axes[idx].set_xlabel(name2)
    axes[idx].set_ylabel(name1)

plt.tight_layout()
plt.show()

### 1.5 Disagreement Distance Distribution

How far apart are annotators when they disagree? (0 = exact match, 1 = adjacent, 4 = maximum)

In [ ]:
all_diffs = []
pair_names = []

for a1, a2 in combinations(annotators, 2):
    mask = df_conflicts[a1].notna() & df_conflicts[a2].notna()
    subset = df_conflicts[mask]
    name1, name2 = a1.split('_')[-1], a2.split('_')[-1]
    
    diff = (subset[a1].map(label_to_int) - subset[a2].map(label_to_int)).abs()
    all_diffs.append(diff)
    pair_names.append(f"{name1}-{name2}")

fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

for i, (diffs, name) in enumerate(zip(all_diffs, pair_names)):
    dist = diffs.value_counts().reindex(range(5), fill_value=0)
    axes[i].bar(dist.index, dist.values, color=['#2ecc71', '#f39c12', '#e74c3c', '#8e44ad', '#2c3e50'])
    axes[i].set_title(f'{name} (n={len(diffs)})')
    axes[i].set_xlabel('Label Distance')
    axes[i].set_xticks(range(5))

axes[0].set_ylabel('Count')
plt.suptitle('Disagreement Distance Distribution', y=1.02)
plt.tight_layout()
plt.show()

# Summary table
print("=== Distance Summary ===\n")
for diffs, name in zip(all_diffs, pair_names):
    print(f"{name}: mean distance = {diffs.mean():.2f}, "
          f"exact = {(diffs == 0).sum()}, "
          f"adjacent = {(diffs == 1).sum()}, "
          f"far (≥2) = {(diffs >= 2).sum()}")

### 1.6 Label Distribution per Annotator (in conflicts)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

for i, col in enumerate(annotators):
    name = col.split('_')[-1]
    counts = df_conflicts[col].dropna().map(lambda x: label_order.index(x) if x in label_order else -1)
    counts = df_conflicts[col].dropna().value_counts().reindex(label_order, fill_value=0)
    axes[i].bar(label_order, counts.values, color='steelblue')
    axes[i].set_title(f'{name} (n={df_conflicts[col].notna().sum()})')
    axes[i].tick_params(axis='x', rotation=45)

axes[0].set_ylabel('Count')
plt.suptitle('Label Distribution per Annotator (conflict articles only)', y=1.02)
plt.tight_layout()
plt.show()

## 2. Merged Dataset Description

In [ ]:
df_merged.info()
print()
print("=== Bias Label Distribution ===")
print(df_merged['bias_label'].value_counts(dropna=False))
print()
print("=== Annotator Distribution ===")
print(df_merged['annotator'].value_counts())
print()
print("=== Publisher Distribution ===")
print(df_merged['publisher'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Label distribution
label_counts = df_merged['bias_label'].value_counts().reindex(label_order)
axes[0].bar(label_order, label_counts.values, color=['#e74c3c', '#f39c12', '#2ecc71', '#3498db', '#9b59b6'])
axes[0].set_title('Bias Label Distribution (merged)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Per-annotator label distribution (stacked)
ct = pd.crosstab(df_merged['bias_label'], df_merged['annotator']).reindex(label_order)
ct.plot(kind='bar', ax=axes[1], stacked=True)
axes[1].set_title('Label Distribution by Annotator')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Annotator')

plt.tight_layout()
plt.show()

## 3. Preprocessing for Modeling

In [ ]:
import unicodedata

df = df_merged[['article_id', 'title', 'body_text', 'bias_label']].copy()

# Unicode normalization (NFC - canonical decomposition + composition)
df['body_text'] = df['body_text'].apply(lambda x: unicodedata.normalize('NFC', x))
df['title'] = df['title'].apply(lambda x: unicodedata.normalize('NFC', x))

df.info()
print()
df.head()